# Day 5: Database Integration & Dashboard Creation
**Hakeem · SparkCity · S.T.E.A.M. convention**

This notebook turns the completed Day 4 handoff into a versioned analytics snapshot, an interactive city-operations dashboard, and an optional PostgreSQL/streaming integration.

**Run All is local-only by default.** No S2 writes, scheduler installation, retention deletion, or external messages occur unless separately enabled. The shared validator and raw loader are unchanged.

The README's production deliverables need deployment verification beyond notebook execution. The historical streaming demo is explicitly a **replay**, not a live city feed. We will investigate convention dates from fiscal evidence after this integration step; this notebook does not select a date.

## 1. Configuration
Use the project's `.venv` kernel after `uv sync --dev`. Restart the kernel when switching from another Spark notebook. Set `DAY4_RUN` to pin a completed run; otherwise use the latest complete run. Leave the database switches false until the team approves the dedicated schema.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, FileLink
from sparkcityx.day5 import (
    METRICS, SCHEMA, digest, validate_bundle, setup_schema,
    publish_bundle, read_publication, retention_candidates,
)
from sparkcityx.day5_pipeline import (
    start_spark, build_snapshot, load_snapshot, replay_alert_stream,
)

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/sparkcityx").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside SparkCity_Capstone.")
DAY4_RUN = None
APPLY_DATABASE = False
INITIALIZE_SCHEMA = False
REPLAY_STREAM = False
if (INITIALIZE_SCHEMA or REPLAY_STREAM) and not APPLY_DATABASE:
    raise ValueError("Schema setup and replay require explicit database opt-in.")
spark = start_spark()
print("Spark:", spark.version)
print("Database publication:", "ENABLED" if APPLY_DATABASE else "disabled — local-only")
print("Target namespace if enabled:", SCHEMA)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/17 00:17:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 4.2.0
Database publication: disabled — local-only
Target namespace if enabled: sparkcity_analytics_hakeem


## 2. Schema and write contract

The star-schema center is a daily observation fact: **run × dataset × metric × date**.
Date and metric dimensions provide calendar and measurement context. Separate alert facts retain sensor IDs, model IDs and reasons; run provenance connects all four earlier notebooks.

| Object | Purpose |
|---|---|
| `dim_date`, `dim_metric` | Calendar and measurement dimensions |
| `pipeline_run`, `fact_daily` | Immutable snapshot identity and daily mean/min/max/coverage |
| `fact_alert` | Day 4 held-out test alerts, not confirmed incidents |
| `publication` | Atomically selects the complete dashboard snapshot |
| `stream_alert`, `stream_batch` | Content-checked historical replay and retry ledger |
| `notification_outbox` | Pending review notifications, not delivery receipts |

The new `sparkcity_analytics_hakeem` namespace does not modify operational `sparkcity` tables.
Same run ID + same payload is a no-op; same ID + changed payload is rejected.
Corrections use a new version. Fact rows and publication-pointer update share one transaction.
Retention defaults to preview and always protects the published run.

In [2]:
ddl = (ROOT / "sql/002_day5_analytics.sql").read_text()
print(ddl)
print("Schema displayed only; no SQL has been executed.")

-- Dedicated namespace: never alters sparkcity raw/source tables.
CREATE SCHEMA IF NOT EXISTS sparkcity_analytics_hakeem;
CREATE TABLE IF NOT EXISTS sparkcity_analytics_hakeem.dim_date (
    day DATE PRIMARY KEY, year INTEGER NOT NULL, month INTEGER NOT NULL CHECK(month BETWEEN 1 AND 12),
    weekday INTEGER NOT NULL CHECK(weekday BETWEEN 0 AND 6)
);
CREATE TABLE IF NOT EXISTS sparkcity_analytics_hakeem.dim_metric (
    dataset TEXT NOT NULL, metric TEXT NOT NULL, interpretation TEXT NOT NULL,
    PRIMARY KEY(dataset, metric)
);
CREATE TABLE IF NOT EXISTS sparkcity_analytics_hakeem.pipeline_run (
    run_id TEXT PRIMARY KEY, payload_sha256 TEXT NOT NULL,
    created_at TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP,
    provenance JSONB NOT NULL
);
CREATE TABLE IF NOT EXISTS sparkcity_analytics_hakeem.fact_daily (
    run_id TEXT REFERENCES sparkcity_analytics_hakeem.pipeline_run ON DELETE CASCADE,
    dataset TEXT NOT NULL, metric TEXT NOT NULL,
    day DATE NOT NULL REFERENCES sparkc

## 3. Build the analytics snapshot with Spark

Verify the Day 3 manifest and Parquet hashes recorded by Day 4. Revalidate required inputs and add publication-local checks for blank/nonfinite values. Aggregate in Spark before collecting compact results.

Each domain retains its own date range; absent dates are not zero-filled. Sensor counts are coverage indicators, not proof of stable physical locations. All daily means describe observations, not city totals. Fiscal `net_per_observation` is the paired revenue-minus-expense difference—not convention profit.

Progress prints once per dataset. This reuses saved inputs and alerts; it does not retrain Day 4 models.

In [3]:
OUTPUT, bundle = build_snapshot(ROOT, spark, DAY4_RUN)
daily = pd.DataFrame(bundle["daily"])
alerts = pd.DataFrame(bundle["alerts"])
display(pd.DataFrame(bundle["provenance"]["audit"]))
display(daily.head())
print("Local snapshot:", OUTPUT)
print("Alert records:", len(alerts))

Day 4 source: 20260914T205935811311Z


traffic: 36,000 rows -> 125 daily groups
air_quality: 36,000 rows -> 375 daily groups
weather: 36,000 rows -> 750 daily groups
energy: 36,000 rows -> 250 daily groups
occupancy: 36,000 rows -> 375 daily groups
fiscal: 36,000 rows -> 375 daily groups
Completed local snapshot: /Users/hakeem/Projects/SparkCity_Capstone/data/processed/day5/20260917T041750168038Z


,dataset,rows,days
0,traffic,36000,125
1,air_quality,36000,375
2,weather,36000,750
3,energy,36000,250
4,occupancy,36000,375
5,fiscal,36000,375


,dataset,metric,day,observations,sensors,mean,minimum,maximum
0,traffic,vehicle_count,2025-01-01,288,288,79.402778,20.0,184.00
1,traffic,avg_speed,2025-01-01,288,288,17.952604,5.0,57.80
2,traffic,vehicle_count,2025-01-02,288,288,79.718750,20.0,186.00
3,traffic,avg_speed,2025-01-02,288,288,17.933299,5.0,54.96
4,traffic,vehicle_count,2025-01-03,288,288,79.597222,20.0,184.00


Local snapshot: /Users/hakeem/Projects/SparkCity_Capstone/data/processed/day5/20260917T041750168038Z
Alert records: 458


## 4. Verify the export and prepare fiscal evidence

Check round-trip integrity, daily row accounting, fiscal arithmetic, all six domains, and alert counts.
Keep fiscal source units and timestamp semantics visible. Date selection is deliberately deferred until the business question and data semantics are confirmed.

In [4]:
restored = load_snapshot(OUTPUT)
assert digest(restored) == digest(bundle)
validate_bundle(restored)
for audit in bundle["provenance"]["audit"]:
    kind = audit["dataset"]
    first_metric = METRICS[kind][0]
    selected = daily[(daily.dataset == kind) & (daily.metric == first_metric)]
    assert int(selected.observations.sum()) == audit["rows"], kind
assert len(alerts) == sum(r["alert_rows"] for r in bundle["provenance"]["anomalies"])
fiscal = daily[daily.dataset == "fiscal"].pivot(index="day", columns="metric", values="mean")
assert np.allclose(fiscal["net_per_observation"], fiscal["revenue"] - fiscal["expense"])
display(fiscal.head(10))
print("Integrity, coverage, fiscal arithmetic and alert-count checks passed.")
print("Fiscal handoff:", OUTPUT / "fiscal_daily.csv")
print("Source units:", bundle["provenance"]["source_units"])
print("Timestamp semantics:", bundle["provenance"]["timestamp_semantics"])

metric,expense,net_per_observation,revenue
day,,,
2025-01-01,94.577917,116.398542,210.976458
2025-01-02,83.389375,111.945104,195.334479
2025-01-03,106.596771,139.647188,246.243958
2025-01-04,91.137500,147.738542,238.876042
2025-01-05,79.892188,180.532083,260.424271
2025-01-06,101.971563,138.292917,240.264479
2025-01-07,96.011146,150.266146,246.277292
2025-01-08,103.491250,102.702292,206.193542
2025-01-09,80.753438,143.358958,224.112396


Integrity, coverage, fiscal arithmetic and alert-count checks passed.
Fiscal handoff: /Users/hakeem/Projects/SparkCity_Capstone/data/processed/day5/20260917T041750168038Z/fiscal_daily.csv
Source units: {'temperature': None, 'avg_speed': None, 'power_consumption': None}
Timestamp semantics: source wall-clock; source timezone unconfirmed


## 5. Interactive dashboard

Open the self-contained HTML in a browser. Choose any of the six datasets, a measurement and a date range; inspect daily coverage and alert reasons, or download the filtered daily records.

The database-backed server below adds 15-second polling, visible stale-state handling and optional browser review notices. Refreshing historical data does not make it current city telemetry. The snapshot dashboard remains usable offline.

In [5]:
display(FileLink(str(OUTPUT / "dashboard.html")))
print("For a database-backed preview AFTER publication:")
print(f'uv run python scripts/serve-day5.py --snapshot "{OUTPUT}"')
print("Then open http://127.0.0.1:8055")

/Users/hakeem/Projects/SparkCity_Capstone/data/processed/day5/20260917T041750168038Z/dashboard.html

For a database-backed preview AFTER publication:
uv run python scripts/serve-day5.py --snapshot "/Users/hakeem/Projects/SparkCity_Capstone/data/processed/day5/20260917T041750168038Z"
Then open http://127.0.0.1:8055


## 6. Optional PostgreSQL batch publication and read-back

**Writes occur only when APPLY_DATABASE is True.** The first deployment additionally needs INITIALIZE_SCHEMA=True and approved schema-creation privileges. Later runs should leave INITIALIZE_SCHEMA=False.

Credentials are read from the environment or ignored `secrets/.env`, never printed. The shared helper requires SSL. Publication is versioned and transactional; it does not overwrite raw sensor data. A read-back checks published counts, then retries the identical bundle to demonstrate no-op behavior.

In [6]:
if APPLY_DATABASE:
    from dotenv import load_dotenv
    from sparkcityx.database import connect_database
    load_dotenv(ROOT / "secrets/.env", override=False)
    with connect_database() as connection:
        if INITIALIZE_SCHEMA:
            setup_schema(connection, ROOT)
        receipt = publish_bundle(connection, bundle)
    with connect_database() as connection:
        published = read_publication(connection)
    if published["run_id"] != bundle["run_id"]:
        raise RuntimeError("Another publisher advanced the snapshot; review before proceeding.")
    assert len(published["daily"]) == len(bundle["daily"])
    assert len(published["alerts"]) == len(bundle["alerts"])
    with connect_database() as connection:
        retry = publish_bundle(connection, bundle)
    assert retry["status"] == "unchanged"
    (OUTPUT / "publication_receipt.json").write_text(json.dumps(receipt, indent=2))
    print("Publication and identical retry verified:", receipt)
else:
    print("SKIPPED: no S2 connection or database writes requested.")

SKIPPED: no S2 connection or database writes requested.


## 7. Optional Structured Streaming alert replay and notifications

With REPLAY_STREAM=True, Spark reads at most one prepared JSON file per microbatch, invokes the transactional PostgreSQL sink and persists a checkpoint. The trigger stops after the available historical files have been processed. Rerunning with the same checkpoint is safe; the database also deduplicates batch IDs and alert IDs.

Each replay alert creates a pending outbox event in the same transaction. Nothing sends email/SMS. Browser notices are an optional review convenience while the database dashboard is open. Future delivery workers need reviewed recipients, authentication, retry and acknowledgement rules.

In [7]:
if REPLAY_STREAM:
    progress = replay_alert_stream(spark, OUTPUT, connect_database)
    display(pd.DataFrame([{"batch_id": r["batchId"], "input_rows": r["numInputRows"]}
                          for r in progress]))
    with connect_database() as connection:
        stream_status = read_publication(connection)
    print("Total unique historical replay alerts:", stream_status["replayed_alerts"])
    print("Pending outbox notifications:", stream_status["pending_notifications"])
else:
    print("SKIPPED: historical streaming replay and notification writes are disabled.")

SKIPPED: historical streaming replay and notification writes are disabled.


## 8. Automation, retention and deployment

The same tested pipeline is callable without notebook execution:
`uv run python scripts/run-day5.py --day4-run <run>`.
Use `--snapshot <path> --apply` to retry a completed publication without regenerating it.

The checked-in LaunchAgent template runs a local-only build hourly when explicitly installed. Review its paths and pinned source first. No scheduler is installed by this notebook. Monitor exit status, logs, source age and manifests; repeated successful runs on unchanged inputs are not fresh observations.

Retention is opt-in, limited to unpublished analytics snapshots older than the chosen age; dimensions and stream/outbox history are retained. Back up before applying deletion. The next cell only previews candidates when the database is already enabled.

In [8]:
if APPLY_DATABASE:
    with connect_database() as connection:
        candidates = retention_candidates(connection, keep_days=90, apply=False)
    print("Retention preview only:", candidates)
else:
    print("Retention not executed; no database connection.")
print("Deployment guide:", ROOT / "docs/day5_workflow.md")
print("Local completion manifest:", OUTPUT / "manifest.json")

Retention not executed; no database connection.
Deployment guide: /Users/hakeem/Projects/SparkCity_Capstone/docs/day5_workflow.md
Local completion manifest: /Users/hakeem/Projects/SparkCity_Capstone/data/processed/day5/20260917T041750168038Z/manifest.json


## 9. Completion and next phase

The local deliverables are the versioned snapshot, dashboard, fiscal export and complete manifest.
Database, stream and deployment status must be reported separately; skipped cells are not passed integration tests.

Before calling this production-ready: verify S2 privileges and query plans, separate read/write accounts, authentication/TLS for shared hosting, backup/restore, scheduler ownership, retention, source freshness targets, and notification delivery. The local server is a development preview.

**Next: fiscal convention-date investigation.** Decide whether we are detecting a past event or choosing a future window. Confirm currency and accounting grain, test calendar patterns and sensor coverage, compare consecutive candidate windows, and corroborate fiscal signals with other datasets. Neither high revenue nor the 27 assumption-based scenarios alone proves an event date.

In [9]:
print("Day 5 local workflow completed.")
print("Dashboard:", OUTPUT / "dashboard.html")
print("Fiscal evidence:", OUTPUT / "fiscal_daily.csv")
print("Database publication requested:", APPLY_DATABASE)
print("Historical replay requested:", REPLAY_STREAM)
# spark.stop() when finished with this kernel.

Day 5 local workflow completed.
Dashboard: /Users/hakeem/Projects/SparkCity_Capstone/data/processed/day5/20260917T041750168038Z/dashboard.html
Fiscal evidence: /Users/hakeem/Projects/SparkCity_Capstone/data/processed/day5/20260917T041750168038Z/fiscal_daily.csv
Database publication requested: False
Historical replay requested: False


26/09/17 01:43:50 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:131)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(Blo